# تدريب صوتين خفيفين (رجل وامرأة) لعمران

النتيجة: نموذج **Piper** واحد فيه صوتين، يشتغل على أي معالج بسرعة، ويتحفظ في مستودعك الخاص
`omran77/omran-voices-piper` على Hugging Face. كل شي مجاني. يشتغل على **Google Colab** أو **Kaggle**.

**على Google Colab (بدون رقم جوال):**
1. https://colab.research.google.com ← **File** ← **Upload notebook** ← اختر هالملف.
2. **Runtime** ← **Change runtime type** ← **T4 GPU** ← Save.
3. يسار الصفحة أيقونة المفتاح 🔑 (Secrets) ← **Add new secret**: الاسم `HF_TOKEN` والقيمة مفتاح Hugging Face من نوع
   **Write** (نفس `HF_WRITE_TOKEN` اللي في Vercel)، وفعّل **Notebook access**.
4. **Runtime** ← **Run all**. لو طلع تحذير إن النوتبوك مب من Google ← **Run anyway**.
   خلّ الصفحة مفتوحة والجهاز ما ينام. لو Colab قطع: **Run all** مرة ثانية، ويكمل من وين وقف.

**على Kaggle (يحتاج تأكيد رقم الجوال):** Settings ← Accelerator **GPU T4 x2** و Internet **On**، و Add-ons ← Secrets
← `HF_TOKEN` (✓)، ثم Save Version ← **Save & Run All (Commit)**. يشتغل بروحه حتى لو سكّرت الصفحة.

كل ساعة يرفع آخر نسخة من النموذج وعيّنات تسمعها في المستودع (`samples/`). تشغيلة ثانية تكمل من آخر نسخة
وتستخدم نفس البيانات.

In [ ]:
# ===== الإعدادات =====
VOICES = [('v5', 'm'), ('v4', 'f')]   # (صوت المتبرع في omran-voices-donors، اسمه في النموذج): رجل، امرأة
SENTENCES_PER_VOICE = 700             # تقريبًا ساعة كلام لكل صوت
MODEL_REPO = 'omran77/omran-voices-piper'
DONORS_REPO = 'omran77/omran-voices-donors'
SENTENCES_URL = 'https://omran-voices.vercel.app/data/sentences.json'
SESSION_HOURS = 11.5                  # أطول جلسة (Kaggle ١٢ ساعة؛ Colab ممكن يقطع قبل، والرفع كل ساعة يحفظ الشغل)
UPLOAD_EVERY_MIN = 30                 # Colab يقطع بدون إنذار: نسخة كل نص ساعة
VOICE_FILE = 'ar_AE-omran-medium'

In [ ]:
import base64, json, os, shutil, subprocess, tarfile, time, urllib.request
from pathlib import Path

T0 = time.time()
W = next(p for p in (Path('/kaggle/working'), Path('/content'), Path.cwd()) if p.exists())
os.chdir(W)


def run(cmd, log=None):
    """Runs a command and shows its output here (the notebook hides a child process's own output)."""
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-40:]
        if log is None:
            print(line, end='')
    if p.wait() != 0:
        # the cause goes into the error itself: it is the red text under the cell
        raise RuntimeError(f'failed ({p.returncode}): {cmd}\n' + ''.join(tail[-15:]))


def out(cmd):
    return subprocess.run(cmd, capture_output=True, text=True, check=True).stdout.strip()


def hours_left():
    return SESSION_HOURS - (time.time() - T0) / 3600


def secret(name):
    """The secret from Kaggle (Add-ons > Secrets) or Colab (the key icon), whichever this is."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except ImportError:
        pass
    try:
        from google.colab import userdata
    except ImportError:
        return os.environ.get(name, '')
    try:
        return userdata.get(name)
    except Exception as e:
        raise RuntimeError('ضيف السر HF_TOKEN من أيقونة المفتاح يسار الصفحة، وفعّل Notebook access') from e


HF_TOKEN = (secret('HF_TOKEN') or '').strip()
assert HF_TOKEN.startswith('hf_'), 'السر HF_TOKEN ناقص أو مب مفتاح Hugging Face'
from huggingface_hub import HfApi, hf_hub_download
api = HfApi(token=HF_TOKEN)
print('Hugging Face account:', api.whoami()['name'])

gpus = [g for g in subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
                                  capture_output=True, text=True).stdout.splitlines() if g.strip()]
print('GPUs:', gpus)
assert gpus, 'ما في كرت: Colab ← Runtime ← Change runtime type ← T4 GPU (أو Kaggle ← Accelerator ← GPU T4 x2)'
if not all('T4' in g for g in gpus):
    print('تنبيه: الأفضل T4 (الكرت P100 قديم على هالمكتبات).')

api.create_repo(MODEL_REPO, private=True, exist_ok=True)
assert api.model_info(MODEL_REPO).private, MODEL_REPO + ' لازم يكون Private'
REPO_FILES = set(api.list_repo_files(MODEL_REPO))

In [ ]:
%%writefile reqA.txt
# What chatterbox-tts 0.1.7 needs, without PyTorch (start.bat installs the GPU build first)
# and without gradio (its demo page, not used here). chatterbox-tts itself goes in with --no-deps.
numpy>=1.24,<2
librosa==0.11.0
s3tokenizer
transformers==5.2.0
diffusers==0.29.0
resemble-perth>=1.0.0
conformer==0.3.2
safetensors==0.5.3
spacy-pkuseg
pykakasi==2.3.0
pyloudnorm
omegaconf
# resemble-perth (the watermark) imports pkg_resources, which setuptools 84 no longer has:
# without it the engine stops at start (PerthImplicitWatermarker is None).
setuptools<81

In [ ]:
%%writefile dialect.json
[
[
"يعطيك العافية",
"يِعْطِيك العافْيَة"
],
[
"ما انتبهت",
"مَا إِنْتَبَهْت"
],
[
"هلا والله",
"هَلا وَالله"
],
[
"هالشغل",
"هَالشُّغُل"
],
[
"بنحلها",
"بِنْحِلّها"
],
[
"المول",
"المُوْلْ"
],
[
"حطيته",
"حَطَّيْتَه"
],
[
"سويته",
"سَوَّيْتَه"
],
[
"اطمّن",
"اطَّمَّن"
],
[
"مسجك",
"مَسِجَك"
],
[
"وايد",
"وايِد"
],
[
"أبشر",
"أَبْشِر"
],
[
"تسلم",
"تِسْلَم"
],
[
"وينك",
"وَيْنَك"
],
[
"أشوف",
"أَشُوف"
],
[
"اطمن",
"اطَّمَّن"
],
[
"مسج",
"مَسِج"
],
[
"كفو",
"كَفُو"
],
[
"شوي",
"شْوَيّ"
]
]

In [ ]:
%%writefile gen_data.py
"""Makes training data for one voice of the light model (Piper).

The voice engine the site uses today (Chatterbox, MIT licence) reads our own sentences in a
donor's voice, from that donor's short sample (about 14 seconds); every clip that passes the
checks below is kept. It runs on Kaggle's free GPU, inside train/omran-voices-piper.ipynb.

  python gen_data.py --ref ref_m.wav --speaker m --part 0 --count 700 \
      --sentences sentences.json --dialect dialect.json --out data [--max-minutes 100]

Writes data/wav/<speaker>_<sentence id>.wav (22 050 Hz mono 16-bit, what Piper trains on) and
appends "<file>\t<text>" to data/<speaker>.tsv. The text is the sentence as written (what the app
will send later), not the respelled one the engine reads. A sentence refused twice by the length
check goes to data/<speaker>.rejected. Run it again and it continues: a sentence in either file is
skipped (the same seeds would only make the same refused clip again).
"""
import argparse
import json
import random
import re
import sys
import time
import wave
from pathlib import Path

SR_OUT = 22050
WORDS_MIN, WORDS_MAX, CHARS_MAX = 5, 15, 200
SHARE = {'gulf': 0.6, 'ar': 0.2, 'msa': 0.2}

# --- the pronunciation dictionary of api/_dialect.js, same rules -------------------------------
# JS: (?<![\p{L}\p{M}])([وفبلك]?)FROM(?![\p{L}\p{M}]). Python's re has no \p{..}: a letter is
# [^\W\d_], and the Arabic marks (harakat, superscript alef) are listed by code point.
_NOT_AFTER = r'(?<![^\W\d_])(?<![ؐ-ًؚ-ٰٟۖ-ۭ])'
_NOT_BEFORE = r'(?![^\W\d_])(?![ؐ-ًؚ-ٰٟۖ-ۭ])'


def dialect_rules(pairs):
    """pairs: [[written, spoken], ...], longest first (as api/_dialect.js exports them)."""
    rules = []
    for src, dst in pairs:
        body = r'\s+'.join(re.escape(w) for w in src.split(' '))
        rules.append((re.compile(_NOT_AFTER + '([وفبلك]?)' + body + _NOT_BEFORE), dst))
    return rules


def apply_dialect(text, rules):
    for rx, dst in rules:
        text = rx.sub(lambda m, d=dst: m.group(1) + d, text)
    return text


# --- which sentences ----------------------------------------------------------------------------
def select(rows, count, part, seed=2026):
    """`count` sentences for voice number `part`: 60% Gulf, 20% general Arabic, 20% formal, 5-15
    words. The same seed shuffles once, and each voice takes its own slice, so two voices never
    read the same sentence and a re-run picks the same ones."""
    by_reg = {}
    for row in rows:
        sid, text, reg = row[0], str(row[1]), row[2]
        if reg not in SHARE or len(text) > CHARS_MAX or '|' in text or '\t' in text:
            continue
        if not WORDS_MIN <= len(text.split()) <= WORDS_MAX:
            continue
        by_reg.setdefault(reg, []).append((sid, text, reg))
    out = []
    for reg, share in SHARE.items():
        pool = sorted(by_reg.get(reg, []))
        random.Random(f'{seed}-{reg}').shuffle(pool)
        n = round(count * share)
        out += pool[part * n:(part + 1) * n]
    return out


# --- is this clip good enough to learn from? ----------------------------------------------------
def letters(text):
    return sum(1 for ch in text if ch.isalpha())


def accept(seconds, text):
    """The engine sometimes stops early or keeps babbling after the sentence. Spoken Arabic runs
    about 0.06-0.09 s per letter: far outside that, the clip is not the sentence."""
    n = letters(text)
    if not n or not 1.0 <= seconds <= 15.0:
        return False
    return 0.035 <= seconds / n <= 0.17


def to_int16(samples):
    import numpy as np
    y = np.asarray(samples, dtype=np.float32)
    peak = float(np.max(np.abs(y))) if y.size else 0.0
    if peak > 0:
        y = y * (0.89 / peak)  # -1 dBFS
    return (np.clip(y, -1, 1) * 32767).astype('<i2').tobytes()


def write_wav(path, pcm16, sr=SR_OUT):
    with wave.open(str(path), 'wb') as w:
        w.setnchannels(1)
        w.setsampwidth(2)
        w.setframerate(sr)
        w.writeframes(pcm16)


def done_ids(tsv):
    if not tsv.exists():
        return set()
    return {line.split('\t', 1)[0] for line in tsv.read_text(encoding='utf-8').splitlines() if line}


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--ref', required=True)
    ap.add_argument('--speaker', required=True)
    ap.add_argument('--part', type=int, required=True)
    ap.add_argument('--count', type=int, default=700)
    ap.add_argument('--sentences', required=True)
    ap.add_argument('--dialect', required=True)
    ap.add_argument('--out', required=True)
    ap.add_argument('--max-minutes', type=float, default=0)
    a = ap.parse_args(argv)

    import librosa
    import torch
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS

    out = Path(a.out)
    (out / 'wav').mkdir(parents=True, exist_ok=True)
    tsv = out / (a.speaker + '.tsv')
    rej = out / (a.speaker + '.rejected')
    have = done_ids(tsv) | done_ids(rej)
    rules = dialect_rules(json.loads(Path(a.dialect).read_text(encoding='utf-8')))
    todo = [s for s in select(json.loads(Path(a.sentences).read_text(encoding='utf-8')), a.count, a.part)
            if f'{a.speaker}_{s[0]}.wav' not in have]
    print(f'[{a.speaker}] {len(have)} clips already, {len(todo)} to make', flush=True)

    model = ChatterboxMultilingualTTS.from_pretrained(device='cuda' if torch.cuda.is_available() else 'cpu')
    with torch.no_grad():
        model.prepare_conditionals(a.ref, exaggeration=0.5)
    t0 = time.time()
    kept = rejected = 0
    with tsv.open('a', encoding='utf-8') as log, rej.open('a', encoding='utf-8') as rejlog:
        for i, (sid, text, reg) in enumerate(todo, 1):
            name = f'{a.speaker}_{sid}.wav'
            if a.max_minutes and time.time() - t0 > a.max_minutes * 60:
                print(f'[{a.speaker}] time is up, stopping', flush=True)
                break
            spoken = apply_dialect(text, rules)
            ok = False
            for attempt in range(2):  # one retry with another seed
                torch.manual_seed(sid * 10 + attempt)
                wav = model.generate(spoken, language_id='ar', exaggeration=0.5, cfg_weight=0.5, temperature=0.8)
                y = wav.squeeze(0).cpu().numpy()
                y, _ = librosa.effects.trim(y, top_db=40)
                if accept(len(y) / model.sr, text):
                    ok = True
                    break
            if not ok:
                rejected += 1
                rejlog.write(name + '\n')
                rejlog.flush()
                continue
            y = librosa.resample(y, orig_sr=model.sr, target_sr=SR_OUT)
            write_wav(out / 'wav' / name, to_int16(y))
            log.write(f'{name}\t{text}\n')
            log.flush()
            kept += 1
            if i % 25 == 0:
                print(f'[{a.speaker}] {i}/{len(todo)}  kept {kept}  rejected {rejected}  {round((time.time() - t0) / 60)} min', flush=True)
    print(f'[{a.speaker}] finished: kept {kept}, rejected {rejected}', flush=True)
    return 0


if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile prep_train.py
"""Turns the clips made by gen_data.py into Piper's training list.

  python prep_train.py --data data --out data/metadata.csv

Reads data/<speaker>.tsv (one per voice) and writes "<file>|<speaker>|<text>", Piper's
multi-speaker format. The text gets the same automatic diacritics (tashkeel) Piper adds when it
speaks Arabic (PiperVoice: use_tashkeel, taskeen_threshold 0.8), so the model trains on exactly
the phonemes it will be given later.
"""
import argparse
import sys
from pathlib import Path

TASKEEN = 0.8


def rows(data_dir):
    for tsv in sorted(Path(data_dir).glob('*.tsv')):
        speaker = tsv.stem
        for line in tsv.read_text(encoding='utf-8').splitlines():
            if not line.strip():
                continue
            name, text = line.split('\t', 1)
            if (Path(data_dir) / 'wav' / name).exists():
                yield name, speaker, text


def build(data_dir, diacritize):
    out, seen = [], set()
    for name, speaker, text in rows(data_dir):
        if name in seen:
            continue
        seen.add(name)
        text = diacritize(text).replace('|', ' ').strip()
        if text:
            out.append(f'{name}|{speaker}|{text}')
    return out


def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument('--data', required=True)
    ap.add_argument('--out', required=True)
    a = ap.parse_args(argv)
    from piper.tashkeel import TashkeelDiacritizer
    tashkeel = TashkeelDiacritizer()
    lines = build(a.data, lambda t: tashkeel(t, taskeen_threshold=TASKEEN))
    Path(a.out).write_text('\n'.join(lines) + '\n', encoding='utf-8')
    speakers = sorted({ln.split('|')[1] for ln in lines})
    print(f'{len(lines)} clips, speakers: {speakers}')
    return 0 if lines else 1


if __name__ == '__main__':
    sys.exit(main())

In [ ]:
%%writefile samples.py
"""A few test sentences from the exported model, one set per voice, to listen to on Hugging Face.

  python samples.py out/ar_AE-omran-medium.onnx out/samples
"""
import json
import sys
import wave
from pathlib import Path

TEXTS = [
    'هلا والله، شحالك اليوم؟',
    'بغيت أروح السوق الحين بس الجو حار وايد.',
    'باجر إن شاء الله نتلاقى في البيت عقب صلاة المغرب.',
]


def main(model, out_dir):
    from piper import PiperVoice, SynthesisConfig
    voice = PiperVoice.load(model)
    speakers = json.loads(Path(model + '.json').read_text(encoding='utf-8')).get('speaker_id_map') or {'voice': None}
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    for name, sid in speakers.items():
        for i, text in enumerate(TEXTS, 1):
            with wave.open(str(out / f'{name}_{i}.wav'), 'wb') as w:
                voice.synthesize_wav(text, w, syn_config=SynthesisConfig(speaker_id=sid))
    print('samples:', sorted(p.name for p in out.glob('*.wav')))
    return 0


if __name__ == '__main__':
    sys.exit(main(sys.argv[1], sys.argv[2]))

In [ ]:
%%writefile train_main.py
"""`python -m piper.train` without its "val_mos" checkpoint (same arguments: fit --data... --model...).

piper-tts 1.8.0 always adds ModelCheckpoint(monitor="val_mos"). With the MOS predictor off
(--model.mos_metric none, or when it cannot be downloaded), val_mos is never logged and Lightning
stops the whole training at the end of the first epoch (MisconfigurationException: could not find
the monitored key). Seen in a local dry run. The val_mel and last.ckpt checkpoints stay.
"""
import piper.train.__main__ as piper_train

piper_train._DEFAULT_CALLBACKS[:] = [
    cb for cb in piper_train._DEFAULT_CALLBACKS if getattr(cb, 'monitor', None) != 'val_mos'
]

if __name__ == '__main__':
    piper_train.main()

In [ ]:
# Two separate Python 3.11 environments: the voice engine (makes the data) and Piper (trains),
# each with the exact versions it was written for, away from the notebook's own packages.
# A file ".ok" marks an environment as finished, so a half-made one is made again.
run('pip install -q uv')
PY_A, PY_B = str(W / 'envA/bin/python'), str(W / 'envB/bin/python')
TORCH = 'torch==2.6.0 torchaudio==2.6.0'
CU = '--index-url https://download.pytorch.org/whl/cu124'
if not Path('envA/.ok').exists():
    run('rm -rf envA && uv venv envA --python 3.11 --seed -q')
    run(f'uv pip install -q --python {PY_A} {TORCH} {CU}')
    run(f'uv pip install -q --python {PY_A} -r reqA.txt {TORCH}')
    run(f'uv pip install -q --python {PY_A} --no-deps chatterbox-tts==0.1.7')
    Path('envA/.ok').touch()
if not Path('envB/.ok').exists():
    # Piper from PyPI (its espeak part comes built). Building it from source needs cmake, and
    # cmake 4 refuses older projects.
    run('rm -rf envB && uv venv envB --python 3.11 --seed -q')
    run(f'uv pip install -q --python {PY_B} {TORCH} {CU}')
    run(f'uv pip install -q --python {PY_B} "piper-tts[train]==1.8.0" cython numpy "setuptools<81" huggingface_hub {TORCH}')
    # The alignment step is a small Cython file the PyPI package leaves out: build it from Piper's source.
    if not Path('piper1-gpl').exists():
        run('git clone -q --depth 1 --branch v1.8.0 https://github.com/OHF-voice/piper1-gpl.git piper1-gpl'
            ' || git clone -q --depth 1 https://github.com/OHF-voice/piper1-gpl.git piper1-gpl')
    mab = Path('mabuild')
    mab.mkdir(exist_ok=True)
    shutil.copy('piper1-gpl/src/piper/train/vits/monotonic_align/core.pyx', mab / 'core.pyx')
    (mab / 'setup.py').write_text('from setuptools import setup, Extension\n'
                                  'from Cython.Build import cythonize\n'
                                  'import numpy\n'
                                  "setup(ext_modules=cythonize([Extension('core', ['core.pyx'], include_dirs=[numpy.get_include()])]))\n")
    run(f'cd mabuild && {PY_B} setup.py -q build_ext --inplace')
    dest = Path(out([PY_B, '-c', 'import piper, os; print(os.path.dirname(piper.__file__))'])) / 'train/vits/monotonic_align/monotonic_align'
    dest.mkdir(parents=True, exist_ok=True)
    (dest / '__init__.py').touch()
    for so in mab.glob('core*.so'):
        shutil.copy(so, dest)
    Path('envB/.ok').touch()
# setuptools 84 has no pkg_resources, which the engine's watermark needs (also for envs made before this line).
for py in (PY_A, PY_B):
    run(f'uv pip install -q --python {py} "setuptools<81"')
run([PY_A, '-c', 'import perth; assert perth.PerthImplicitWatermarker, "watermark not importable";'
                 ' from chatterbox.mtl_tts import ChatterboxMultilingualTTS; print("voice engine ok")'])
run([PY_B, '-c', 'from piper.train.vits.monotonic_align import maximum_path; from piper.phonemize_espeak import EspeakPhonemizer;'
                 ' print("piper ok", EspeakPhonemizer().phonemize("ar", "هلا والله"))'])

In [ ]:
# The donors' samples (private dataset) and our sentences.
def donors():
    names = [f for f in api.list_repo_files(DONORS_REPO, repo_type='dataset') if f.endswith('.json')]
    name = 'voices.json' if 'voices.json' in names else names[0]
    return {v['id']: v for v in json.loads(Path(hf_hub_download(DONORS_REPO, name, repo_type='dataset', token=HF_TOKEN)).read_text())}

dv = donors()
for vid, spk in VOICES:
    v = dv[vid]
    assert v.get('consent') is True, vid + ': no consent'
    if v.get('data'):
        Path(f'ref_{spk}.wav').write_bytes(base64.b64decode(v['data']))
    else:
        shutil.copy(hf_hub_download(DONORS_REPO, v['file'], repo_type='dataset', token=HF_TOKEN), f'ref_{spk}.wav')
    print(spk, '<-', vid, v.get('gender'), v.get('seconds'), 's')
urllib.request.urlretrieve(SENTENCES_URL, 'sentences.json')
print(len(json.loads(Path('sentences.json').read_text())), 'sentences')

In [ ]:
# Step 1: the engine reads ~700 sentences in each voice (one voice per GPU). What is made is saved
# in the repo every 20 minutes, so a run cut off (Colab) continues where it stopped.
DATA_TAR = 'data/train-data.tar'


def save_data():
    with tarfile.open('train-data.tar', 'w') as t:
        t.add('data')
    api.upload_file(path_or_fileobj='train-data.tar', path_in_repo=DATA_TAR, repo_id=MODEL_REPO, commit_message='training data')


def clips(spk, ext='tsv'):
    p = Path(f'data/{spk}.{ext}')
    return len(p.read_text(encoding='utf-8').splitlines()) if p.exists() else 0


if DATA_TAR in REPO_FILES and not Path('data').exists():
    with tarfile.open(hf_hub_download(MODEL_REPO, DATA_TAR, token=HF_TOKEN)) as t:
        t.extractall('.')
    print('data from an earlier run:', {spk: (clips(spk), clips(spk, 'rejected')) for _, spk in VOICES})

# a voice is done when every one of its sentences was tried (kept, or refused by the length check)
todo = [(k, spk) for k, (vid, spk) in enumerate(VOICES) if clips(spk) + clips(spk, 'rejected') < SENTENCES_PER_VOICE - 2]
running, last_save = [], time.time()
while todo or running:
    while todo and len(running) < len(gpus):
        k, spk = todo.pop(0)
        gpu = min(set(range(len(gpus))) - {g for _, _, g in running})
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), HF_TOKEN=HF_TOKEN)
        cmd = [PY_A, 'gen_data.py', '--ref', f'ref_{spk}.wav', '--speaker', spk, '--part', str(k),
               '--count', str(SENTENCES_PER_VOICE), '--sentences', 'sentences.json',
               '--dialect', 'dialect.json', '--out', 'data', '--max-minutes', '150']
        running.append((spk, subprocess.Popen(cmd, env=env, stdout=open(f'gen_{spk}.log', 'a'), stderr=subprocess.STDOUT), gpu))
    time.sleep(120)
    for item in list(running):
        spk, p, _ = item
        lines = Path(f'gen_{spk}.log').read_text(errors='replace').splitlines()
        print(time.strftime('%H:%M'), lines[-1] if lines else '...')
        if p.poll() is not None:
            running.remove(item)
            if p.returncode != 0:
                print('\n'.join(lines[-30:]))
                raise RuntimeError(spk + ' failed')
    if time.time() - last_save > 20 * 60 or not (todo or running):
        save_data()
        last_save = time.time()
for _, spk in VOICES:
    rows = Path(f'data/{spk}.tsv').read_text(encoding='utf-8').splitlines()
    secs = sum((Path('data/wav') / ln.split('\t')[0]).stat().st_size - 44 for ln in rows) / 2 / 22050
    print(f'{spk}: {len(rows)} clips, {secs / 60:.0f} minutes')

In [ ]:
# Step 2: Piper learns both voices in one model. It starts from the Arabic voice "kareem"
# (or from the last saved version, on a second run), and stops in time to export and upload.
run([PY_B, 'prep_train.py', '--data', 'data', '--out', 'data/metadata.csv'])
if 'ckpt/last.ckpt' in REPO_FILES:
    warm = hf_hub_download(MODEL_REPO, 'ckpt/last.ckpt', token=HF_TOKEN)
    print('continuing from the last saved version')
else:
    base = [f.path for f in HfApi().list_repo_tree('rhasspy/piper-checkpoints', path_in_repo='ar/ar_JO/kareem/medium',
                                                    repo_type='dataset') if f.path.endswith('.ckpt')]
    warm = hf_hub_download('rhasspy/piper-checkpoints', base[0], repo_type='dataset')
    print('starting from', base[0])

left = hours_left() - 0.6  # export and upload need ~30 minutes
assert left > 0.5, 'not enough time left in this session'
hh, mm = int(left), int((left - int(left)) * 60)
Path('out').mkdir(exist_ok=True)
cmd = [PY_B, 'train_main.py', 'fit',
       '--data.voice_name', VOICE_FILE, '--data.csv_path', 'data/metadata.csv', '--data.audio_dir', 'data/wav',
       '--model.sample_rate', '22050', '--data.espeak_voice', 'ar', '--data.cache_dir', 'cache',
       '--data.trim_silence', 'false',  # gen_data.py already trimmed; Piper's own trim can leave a clip too short to train on
       '--data.config_path', 'out/config.json', '--data.batch_size', '16', '--data.num_workers', '2',
       '--model.num_speakers', str(len(VOICES)), '--model.warmstart_ckpt', warm, '--model.mos_metric', 'none',
       '--trainer.accelerator', 'gpu', '--trainer.devices', '1', '--trainer.max_time', f'00:{hh:02d}:{mm:02d}:00',
       '--trainer.default_root_dir', 'runs']
print(' '.join(cmd))
train = subprocess.Popen(cmd, stdout=open('train.log', 'w'), stderr=subprocess.STDOUT)


def export_and_upload(final=False):
    cks = sorted(W.glob('runs/lightning_logs/version_*/checkpoints/last.ckpt'), key=lambda p: p.stat().st_mtime)
    if not cks:
        print('no checkpoint yet')
        return
    shutil.copy(cks[-1], 'snap.ckpt')
    try:
        run([PY_B, '-m', 'piper.train.export_onnx', '--checkpoint', 'snap.ckpt', '--output-file', f'out/{VOICE_FILE}.onnx'], log=False)
        shutil.copy('out/config.json', f'out/{VOICE_FILE}.onnx.json')
        run([PY_B, 'samples.py', f'out/{VOICE_FILE}.onnx', 'out/samples'], log=False)
    except RuntimeError as e:
        print('export failed this time:', e)
        return
    api.upload_folder(folder_path='out', repo_id=MODEL_REPO, commit_message='model' + (' (final)' if final else ''))
    api.upload_file(path_or_fileobj='snap.ckpt', path_in_repo='ckpt/last.ckpt', repo_id=MODEL_REPO, commit_message='checkpoint')
    print(time.strftime('%H:%M'), 'uploaded to', MODEL_REPO)


last_up = time.time()
while train.poll() is None:
    time.sleep(300)
    lines = Path('train.log').read_text(errors='replace').splitlines()
    print(time.strftime('%H:%M'), (lines[-1] if lines else '...')[-200:])
    if time.time() - last_up > UPLOAD_EVERY_MIN * 60:
        export_and_upload()
        last_up = time.time()
print(Path('train.log').read_text(errors='replace')[-3000:])
export_and_upload(final=True)
print('done in', round((time.time() - T0) / 3600, 1), 'hours')